In [ ]:
from scipy.io import wavfile
import numpy as np
import matplotlib.pyplot as plt
import glob

In [ ]:
# Compute averaged frequency response (magnitude in dB) for a single wav file
FFT_SIZE = 16384
CHANNEL_IDX = 7  # 8th channel (0-indexed)

def compute_measured_response(wav_path):
    rate, data = wavfile.read(wav_path)

    if len(data.shape) == 1:
        print(f"Warning: {wav_path} is mono, skipping")
        return None

    channel_data = data[:, CHANNEL_IDX]
    num_ffts = len(channel_data) // FFT_SIZE
    if num_ffts == 0:
        print(f"Warning: {wav_path} too short, skipping")
        return None

    fft_sum = np.zeros(FFT_SIZE // 2 + 1)
    for i in range(num_ffts):
        start_idx = i * FFT_SIZE
        segment = channel_data[start_idx:start_idx + FFT_SIZE]
        fft_sum += np.abs(np.fft.rfft(segment))

    avg_magnitude = fft_sum / num_ffts
    magnitude_db = 20 * np.log10(avg_magnitude + 1e-10)
    magnitude_db -= np.max(magnitude_db)
    freqs = np.fft.rfftfreq(FFT_SIZE, 1/rate)

    return {'freqs': freqs, 'magnitude_db': magnitude_db, 'num_ffts': num_ffts}

In [ ]:
import re

# Parse coefficient files to extract theoretical filter coefficients
def parse_coef_file(filepath, coef_name):
    """Extract coefficients from .xc file for a given array name (debug arrays are decimal)"""
    with open(filepath, 'r') as f:
        content = f.read()

    # Find the array definition
    pattern = rf'const int {coef_name}\[\d+\] = \{{([^}}]+)\}};'
    match = re.search(pattern, content, re.MULTILINE | re.DOTALL)

    if not match:
        return None

    # Extract decimal values (debug arrays are already in decimal, not hex)
    decimal_values = re.findall(r'-?\d+', match.group(1))

    # Convert to integers
    coefs = [int(val) for val in decimal_values]

    return np.array(coefs)

# Each entry is a self-contained filter set: model coefficients, its matching
# impulse-response wav (None until generated), and the color/label to plot it with.
filter_configs = [
    {
        'label': "Primo EM215 (version 318/9)",
        'coef_file': 'fir_coefs_cascade.xc',
        'coef_name_2': 'g_second_to_third_fir47_48kHz_debug',
        'coef_name_3': 'g_third_to_output_fir47_36kHz_to_48kHz_debug',
        'num_taps': 48,
        'upsample_factor': 2,
        'color': "purple",
        'impulse_response_wav': "wav/SCP-00-CEX-3804_primo-em215_0x319.wav",
    },
    {
        'label': "Primo EM215 (version 320/1)",
        'coef_file': 'fir_coefs_cascade.xc',
        'coef_name_2': 'g_second_to_third_fir47_48kHz_debug',
        'coef_name_3': 'g_third_to_output_fir47_24kHz_debug',
        'num_taps': 48,
        'upsample_factor': 2,
        'color': "red",
        'impulse_response_wav': "wav/SCP-00-CEX-3804_primo-em215_0x320.wav",
    },
    {
        'label': "Infineon IM72D128",
        'coef_file': 'fir_coefs_cascade.xc',
        'coef_name_2': 'g_second_to_third_fir47_48kHz_debug',
        'coef_name_3': 'g_third_to_output_fir47_8kHz_to_24kHz_debug',
        'num_taps': 48,
        'upsample_factor': 2,
        'color': "green",
        'impulse_response_wav': "wav/SCP-00-CEX-3804_im72d128v01_0x320.wav",
    },
    {
        'label': "Vesper VM3000",
        'coef_file': 'fir_coefs_cascade.xc',
        'coef_name_2': 'g_second_to_third_fir47_48kHz_debug',
        'coef_name_3': 'g_third_to_output_fir47_8kHz_to_20kHz_debug',
        'num_taps': 48,
        'upsample_factor': 2,
        'color': "blue",
        'impulse_response_wav': "wav/SCP-00-CEX-3804_vm3000_0x320.wav",
    },
    # {
    #     # 384 kHz -> fir2_debug (16-tap) -> decimate 4:1 -> 96 kHz -> fir3_div_2_debug (64-tap) -> decimate 2:1 -> 48 kHz
    #     # fir3 runs at 96 kHz = 384/4, so upsample by 4 to combine at 384 kHz input rate
    #     'label': 'legacy',
    #     'coef_file': 'coefs/legacy_coefs.xct',
    #     'coef_name_2': 'fir2_debug',
    #     'coef_name_3': 'fir3_div_2_debug',
    #     'num_taps': 16,
    #     'upsample_factor': 4,
    #     'color': "grey",
    #     'impulse_response_wav': None,
    # },
]

# Load theoretical (model) coefficients into each filter_config entry
for config in filter_configs:
    coefs_2 = parse_coef_file(config['coef_file'], config['coef_name_2'])
    coefs_3 = None
    if config['coef_name_3']:
        coefs_3 = parse_coef_file(config['coef_file'], config['coef_name_3'])

    if (coefs_2 is not None) and (coefs_3 is not None):
        upsample = config['upsample_factor']
        # H_total(z) = H2(z) * H3(z^upsample)
        coefs_3_upsampled = np.zeros(len(coefs_3) * upsample)
        coefs_3_upsampled[::upsample] = coefs_3

        config['model_coefs'] = np.convolve(coefs_2, coefs_3_upsampled)
        print(f"Loaded cascaded {config['label']}: {len(coefs_2)} + {len(coefs_3)} coefficients -> {len(config['model_coefs'])} taps (upsampled x{upsample})")
    elif coefs_2 is not None:
        config['model_coefs'] = coefs_2
        print(f"Loaded {config['label']}: {len(coefs_2)} coefficients")
    else:
        config['model_coefs'] = None
        print(f"Warning: Could not load {config['label']}")

In [ ]:
# Compute theoretical (model) frequency responses at 384 kHz (stage 2 input rate),
# and measured responses for any filter_configs entry with an available wav file.
STAGE2_INPUT_RATE = 384000  # Hz
FFT_SIZE_THEORY = 8192

for config in filter_configs:
    coefs = config['model_coefs']
    if coefs is not None:
        padded = np.zeros(FFT_SIZE_THEORY)
        padded[:len(coefs)] = coefs

        magnitude = np.abs(np.fft.rfft(padded))
        magnitude_db = 20 * np.log10(magnitude + 1e-10)
        magnitude_db -= np.max(magnitude_db)
        freqs = np.fft.rfftfreq(FFT_SIZE_THEORY, 1/STAGE2_INPUT_RATE)

        config['model_response'] = {'freqs': freqs, 'magnitude_db': magnitude_db}

    config['measured_response'] = None
    if config['impulse_response_wav']:
        config['measured_response'] = compute_measured_response(config['impulse_response_wav'])

print(f"Computed model responses for {sum(c['model_response'] is not None for c in filter_configs)} filters, "
      f"measured responses for {sum(c['measured_response'] is not None for c in filter_configs)} filters")

In [ ]:
# Plot model (dashed) vs measured (solid) responses for each filter_configs entry
plt.figure(figsize=(16, 10))

for config in filter_configs:
    color = config['color']

    # Model: thick, semi-transparent line underneath so the thin measured
    # trace on top is visible wherever the two curves coincide.
    if config['model_response'] is not None:
        data = config['model_response']
        plt.plot(data['freqs'] / 1000, data['magnitude_db'],
                 label=f"{config['label']} (model)", linewidth=3, color=color,
                 linestyle='-', alpha=0.35, solid_capstyle='round')

    # Measured: thin dashed line with dense markers, drawn on top of the model.
    if config['measured_response'] is not None:
        data = config['measured_response']
        plt.plot(data['freqs'] / 1000, data['magnitude_db'],
                 label=f"{config['label']} (measured)", linewidth=1.2, color=color,
                 linestyle='--', marker='o', markevery=0.01, markersize=3)

plt.xlabel('Frequency (kHz)', fontsize=14)
plt.ylabel('Magnitude (dB)', fontsize=14)
plt.title('Model vs Measured Filter Response', fontsize=16)
plt.grid(True, alpha=0.3)
plt.legend(loc='upper right', fontsize=16)
plt.xlim(0, 96)
plt.xticks(np.arange(0, 100, 4))
plt.ylim(-120, 3)
plt.yticks(np.arange(-120, 6, 6))
plt.axhline(y=-3, color='r', linestyle=':', alpha=0.5, linewidth=1)
plt.axvline(x=48, color='k', linestyle='--', alpha=0.5, linewidth=1)
plt.tight_layout()
plt.show()

In [ ]:
# Zoomed-in passband view: DC to 12 kHz, -3 to +1 dB
plt.figure(figsize=(16, 10))

for config in filter_configs:
    color = config['color']

    if config['model_response'] is not None:
        data = config['model_response']
        plt.plot(data['freqs'] / 1000, data['magnitude_db'],
                 label=f"{config['label']} (model)", linewidth=3, color=color,
                 linestyle='-', alpha=0.35, solid_capstyle='round')

    if config['measured_response'] is not None:
        data = config['measured_response']
        plt.plot(data['freqs'] / 1000, data['magnitude_db'],
                 label=f"{config['label']} (measured)", linewidth=1.2, color=color,
                 linestyle='--', marker='o', markevery=0.01, markersize=3)

plt.xlabel('Frequency (kHz)', fontsize=14)
plt.ylabel('Magnitude (dB)', fontsize=14)
plt.title('Model vs Measured Filter Response (Passband Zoom)', fontsize=16)
plt.grid(True, alpha=0.3)
plt.legend(loc='lower right', fontsize=16)
plt.xlim(0, 1)
plt.xticks(np.arange(0, 1.1, 0.1))
plt.ylim(-3, 1)
plt.yticks(np.arange(-3, 1.5, 0.5))
plt.axhline(y=-3, color='r', linestyle=':', alpha=0.5, linewidth=1)
plt.tight_layout()
plt.show()